# Projeto II — Sprint II | Testes de hipótese

**Tema:** tempo de carregamento de caminhões com grãos.

Faça upload das duas planilhas Excel quando solicitado. A Planilha 1 contém os dados anteriores à intervenção (referência) e a Planilha 2 contém os dados posteriores.

A comparação antes/depois é observacional e, isoladamente, não comprova causalidade.

## 1. Estatísticas descritivas e definição do evento

A Questão 5 da Sprint 1 usa o intervalo [20,43] minutos para estimar p₀. O relatório também registra a faixa-alvo [20,30] para outras questões. Não misture intervalos diferentes no mesmo teste. Este notebook utiliza [20,43] em ambos os períodos para manter o evento consistente; se a orientação específica da Sprint 2 exigir outro intervalo, altere `a` e `b` após conferir o enunciado.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from google.colab import files
from io import BytesIO

COLUNA = "Tempo de carregamento (min)"

print("Envie a Planilha 1 (antes).")
u1 = files.upload()
df1 = pd.read_excel(BytesIO(next(iter(u1.values()))))
print("Envie a Planilha 2 (depois).")
u2 = files.upload()
df2 = pd.read_excel(BytesIO(next(iter(u2.values()))))

for nome, df in [("Planilha 1", df1), ("Planilha 2", df2)]:
    if COLUNA not in df.columns:
        raise ValueError(f"{nome}: coluna {COLUNA!r} não encontrada. Colunas disponíveis: {df.columns.tolist()}")

antes = pd.to_numeric(df1[COLUNA], errors="coerce").dropna()
depois = pd.to_numeric(df2[COLUNA], errors="coerce").dropna()
print("n antes:", len(antes), "| n depois:", len(depois))

In [ ]:
a, b = 20, 43
n1, n2 = len(antes), len(depois)
mu0, s0 = antes.mean(), antes.std(ddof=1)

# p0: probabilidade sob a normal ajustada aos dados anteriores (Questão 5).
p0 = stats.norm.cdf(b, loc=mu0, scale=s0) - stats.norm.cdf(a, loc=mu0, scale=s0)

# p-hat posterior: frequência observada no mesmo intervalo [a,b].
sucessos = int(depois.between(a, b, inclusive="both").sum())
phat = sucessos / n2

# Apenas referência: proporção posterior na faixa [20,30] mencionada no relatório.
phat_20_30 = depois.between(20, 30, inclusive="both").mean()

print(f"Intervalo usado no teste: [{a}, {b}] min")
print(f"p0 = {p0:.6f}")
print(f"p-hat = {sucessos}/{n2} = {phat:.4%}")
print(f"Referência p-hat em [20,30] = {phat_20_30:.2%}")

## 2. Hipóteses e estatística de teste

Como o objetivo é verificar aumento da proporção de carregamentos no intervalo definido:

- H₀: p = p₀
- H₁: p > p₀

O teste é unilateral à direita.

In [ ]:
ep0 = np.sqrt(p0 * (1 - p0) / n2)
z_calc = (phat - p0) / ep0
p_valor = 1 - stats.norm.cdf(z_calc)

print(f"Erro-padrão sob H0 = {ep0:.6f}")
print(f"z calculado = {z_calc:.4f}")
print(f"p-valor unilateral à direita = {p_valor:.6g}")
print(f"Sucessos esperados sob H0 = {n2*p0:.3f}")
print(f"Fracassos esperados sob H0 = {n2*(1-p0):.3f}")

## 3. Decisão para α = 5% e α = 1%

In [ ]:
def testar_proporcao(alpha):
    zcrit = stats.norm.ppf(1 - alpha)
    rejeita = z_calc >= zcrit
    print(f"\\nα = {alpha:.0%}")
    print(f"Valor crítico: {zcrit:.4f}")
    print(f"Região crítica: z ≥ {zcrit:.4f}")
    print(f"z calculado: {z_calc:.4f}")
    print("Decisão:", "REJEITAR H0" if rejeita else "NÃO REJEITAR H0")
    print("Conclusão:", ("há evidência de aumento da proporção" if rejeita
                        else "não há evidência suficiente de aumento da proporção"))
    return zcrit

zcrit_5 = testar_proporcao(.05)
zcrit_1 = testar_proporcao(.01)

## 4. Gráficos das regiões críticas

In [ ]:
def grafico_proporcao(alpha):
    zcrit = stats.norm.ppf(1 - alpha)
    x = np.linspace(-4, 4, 1000)
    y = stats.norm.pdf(x)
    plt.figure(figsize=(9, 4.5))
    plt.plot(x, y, label="Normal padrão")
    plt.fill_between(x, 0, y, where=x >= zcrit, alpha=.3, label="Região de rejeição")
    plt.axvline(zcrit, ls="--", label=f"z crítico = {zcrit:.3f}")
    plt.axvline(z_calc, ls="-.", label=f"z calculado = {z_calc:.3f}")
    plt.title(f"Teste unilateral à direita — α={alpha:.0%}")
    plt.xlabel("z"); plt.ylabel("Densidade"); plt.legend(); plt.grid(alpha=.2); plt.show()

grafico_proporcao(.05)
grafico_proporcao(.01)

## 5. Verificação metodológica e conclusão

A aproximação normal do teste z exige verificar as frequências esperadas sob H₀. Se `n*p0` ou `n*(1-p0)` for pequeno, a aproximação pode ser inadequada; confirme com o docente se é necessário apresentar um teste binomial exato como análise complementar.

Compare p₀ e p̂ usando exatamente o mesmo intervalo. **Não rejeitar H₀ não prova igualdade.** A comparação antes/depois é observacional e não demonstra causalidade.